[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_53_Production_Deployment.ipynb)

# Lesson 53 — Phase 5 · Production Deployment

**Phase 5 roadmap (OSS auto_researcher_v2)**

| # | Lesson | Status |
|---|--------|--------|
| L51 | Architecture & Scaffold | ✅ |
| L52 | Advanced Retrieval: Beyond Naive RAG | ✅ |
| **L53** | **Production Deployment** | **← you are here** |
| L54 | Developer Experience (CLI, SDK, docs) | 🔜 |
| L55 | Capstone: Ship It (PyPI + GitHub release) | 🔜 |

---

## What you'll build today

The gap between *"runs in Colab"* and *"runs in production"* is wide.  
A notebook cell crashing at 3 AM is annoying. A production API crashing at 3 AM pages someone.

Today we close that gap for `auto_researcher_v2`. By the end of this lesson you'll have:

- A **multi-stage Docker image** (~120 MB, non-root, health-checked)
- A **hardened FastAPI app** with request-ID tracing, rate limiting, and auth
- A **production config system** that fails fast on missing secrets
- **Structured JSON logging** so Datadog / Cloud Logging can parse every line
- **Health / readiness / liveness** endpoints that Kubernetes and load balancers use
- **Graceful shutdown** so in-flight requests drain before the process exits
- **Fly.io and Railway configs** — two free/cheap deploy targets
- A **BudgetGuard middleware** that enforces a daily spend cap
- An **async load tester** to find your breaking point before your users do

**This lesson is mostly write-files-and-read-them.** The actual server won't bind in Colab (ports are firewalled), but every snippet is identical to what you'd run locally or in CI.

## 1 — What "production-ready" actually means

Here is a minimal checklist. Engineers argue about the long tail; these 12 items are non-negotiable:

| # | Property | What breaks if you skip it |
|---|----------|---------------------------|
| 1 | **Secrets never in source** | Leaked API keys, fired engineers |
| 2 | **Config fails fast at startup** | Crashes 10 minutes into handling traffic |
| 3 | **Structured logs** | Can't search, alert, or debug production |
| 4 | **Request IDs** | Can't correlate logs across services |
| 5 | **Health endpoints** | Load balancer sends traffic to dead pods |
| 6 | **Graceful shutdown** | In-flight requests return 502 on deploy |
| 7 | **Rate limiting** | One bad client takes down everyone |
| 8 | **Auth** | Anyone can call your LLM and you pay the bill |
| 9 | **Non-root Docker user** | Container escape = root on host |
| 10 | **Resource limits** | OOM kill cascades across pod |
| 11 | **Budget cap** | $10K bill from a runaway loop |
| 12 | **Load-tested capacity** | Surprise traffic spike = outage |

We'll implement all 12 today.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install fastapi "uvicorn[standard]" httpx pydantic-settings python-dotenv \
             anthropic nest_asyncio rich aiohttp -q

import os, json, time, uuid, asyncio, sqlite3, logging, textwrap, pathlib
from datetime import datetime, date
from typing import Optional, Dict, Any, List, Callable, AsyncGenerator
from dataclasses import dataclass, field
from contextlib import asynccontextmanager
from collections import defaultdict

import nest_asyncio
nest_asyncio.apply()

# ── API key ───────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    print("⚠️  Set ANTHROPIC_API_KEY manually if you need live LLM calls.")

WORKSPACE = pathlib.Path("/content/auto_researcher_v2")
WORKSPACE.mkdir(parents=True, exist_ok=True)
print(f"Workspace: {WORKSPACE}")

## 2 — Multi-stage Docker: small, secure, cacheable

### Why multi-stage?

```
Single-stage build                    Multi-stage build
──────────────────                    ────────────────────────────
python:3.11 base  ≈ 1.0 GB           Stage 1 (builder)
+ gcc / build tools                     python:3.11-slim + build deps
+ all dev packages                      compile wheels → /wheels/
+ source + tests + docs                ↓
= final image ≈ 1.4 GB               Stage 2 (runtime)            
                                        python:3.11-slim (no gcc)
                                        COPY --from=builder /wheels/
                                        pip install --no-index /wheels/
                                        = final image ≈ 120 MB
```

**Three rules for a production Dockerfile:**
1. `COPY pyproject.toml` first → Docker cache layer survives code-only changes
2. Run as a **non-root user** (UID 1000) — if something escapes the container it doesn't get host root
3. Declare a `HEALTHCHECK` — Docker and ECS/Fly.io use it to know when the container is alive

### Layer cache strategy

```
Layer 1: FROM python:3.11-slim          ← almost never invalidated
Layer 2: RUN apt-get install ...        ← invalidated if OS deps change
Layer 3: COPY pyproject.toml .          ← invalidated if deps change
Layer 4: RUN pip install ...            ← invalidated if deps change
Layer 5: COPY src/ .                    ← invalidated every code push
```

Putting `COPY pyproject.toml` **before** `COPY src/` means a code-only change
skips the slow `pip install` layer entirely. This matters a lot in CI.

In [ ]:
# ── Multi-stage Dockerfile ────────────────────────────────────────────────────
DOCKERFILE = """\
# ── Stage 1: builder ──────────────────────────────────────────────────────────
FROM python:3.11-slim AS builder

WORKDIR /build

# System deps needed only at build time (wheels that compile C extensions)
RUN apt-get update && apt-get install -y --no-install-recommends \\
        build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Cache pip layer independently from source code
COPY pyproject.toml .
RUN pip install --upgrade pip wheel \\
 && pip wheel --no-cache-dir --wheel-dir /wheels .

# ── Stage 2: runtime ─────────────────────────────────────────────────────────
FROM python:3.11-slim AS runtime

# Non-root user (UID 1000 is conventional)
RUN groupadd -g 1000 appgroup && useradd -u 1000 -g appgroup -m appuser

WORKDIR /app

# Copy pre-built wheels from builder stage (no gcc needed here)
COPY --from=builder /wheels /wheels
RUN pip install --no-index --find-links /wheels auto-researcher \\
 && rm -rf /wheels

# Copy application source AFTER deps (layer cache advantage)
COPY auto_researcher/ ./auto_researcher/

# Switch to non-root before EXPOSE and CMD
USER appuser

EXPOSE 8000

# Docker native health check — also used by Fly.io / ECS
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD python -c "import httpx; httpx.get('http://localhost:8000/live').raise_for_status()"

# Exec-form CMD so SIGTERM reaches uvicorn (not a shell wrapper)
CMD ["uvicorn", "auto_researcher.api:app", \\
     "--host", "0.0.0.0", \\
     "--port", "8000", \\
     "--workers", "1", \\
     "--timeout-graceful-shutdown", "30"]
"""

dockerfile_path = WORKSPACE / "Dockerfile"
dockerfile_path.write_text(DOCKERFILE)
print("✅ Dockerfile written")
print()
print(dockerfile_path.read_text())

## 3 — Production config: fail fast, never guess

### The problem with `os.getenv("KEY", "")`

```python
# BAD — silently continues with empty string
api_key = os.getenv("ANTHROPIC_API_KEY", "")
client = Anthropic(api_key=api_key)  # works, but first real call explodes
```

The failure happens *inside the request handler*, 30 seconds after deploy,  
while real traffic is already flowing.

### The fix: Pydantic Settings with startup validation

```
startup                    request handler
───────────────────────    ────────────────────────
ProductionConfig()    →    config.anthropic_api_key
  reads ALL env vars         (already validated, never empty)
  raises ValidationError
  → process exits immediately
  → orchestrator restarts with clear error in logs
  → NO traffic served with bad config
```

**Rule: if your app can't work without a value, that value must be required (no default).**

In [ ]:
# ── Production config with Pydantic Settings ──────────────────────────────────
CONFIG_PY = """
import os
from typing import Optional, Literal
from pydantic import field_validator, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class ProductionConfig(BaseSettings):
    """All config from environment variables.
    
    Usage:
        config = ProductionConfig()            # reads env at import time
        config = ProductionConfig(_env_file=".env")  # also reads .env file
    """
    model_config = SettingsConfigDict(
        env_prefix="AR_",   # AR_ANTHROPIC_API_KEY, AR_API_KEY, etc.
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=False,
    )

    # ── Required secrets (no default = must be set) ───────────────────────────
    anthropic_api_key: str
    api_key: str           # our own API key clients must send

    # ── Optional with sensible defaults ──────────────────────────────────────
    host: str = "0.0.0.0"
    port: int = 8000
    environment: Literal["development", "staging", "production"] = "production"
    log_level: Literal["DEBUG", "INFO", "WARNING", "ERROR"] = "INFO"

    # ── Budget guardrails ─────────────────────────────────────────────────────
    daily_budget_usd: float = 10.0      # max $ per calendar day
    per_request_budget_usd: float = 0.10  # max $ per single request

    # ── Rate limiting ─────────────────────────────────────────────────────────
    rate_limit_rpm: int = 60            # requests per minute per IP

    # ── Model selection ───────────────────────────────────────────────────────
    orchestrator_model: str = "claude-sonnet-4-5"
    worker_model: str = "claude-haiku-4-5"

    # ── Retrieval ─────────────────────────────────────────────────────────────
    retrieval_top_k: int = 5
    enable_hybrid_search: bool = True

    # ── Validators ────────────────────────────────────────────────────────────
    @field_validator("anthropic_api_key")
    @classmethod
    def key_must_look_like_anthropic(cls, v: str) -> str:
        if not v.startswith("sk-ant-"):
            raise ValueError(
                "AR_ANTHROPIC_API_KEY must start with 'sk-ant-'. "
                "Check your environment variables."
            )
        return v

    @field_validator("api_key")
    @classmethod
    def api_key_min_length(cls, v: str) -> str:
        if len(v) < 16:
            raise ValueError("AR_API_KEY must be at least 16 characters")
        return v

    @model_validator(mode="after")
    def budget_sanity(self) -> "ProductionConfig":
        if self.per_request_budget_usd > self.daily_budget_usd:
            raise ValueError(
                "per_request_budget_usd cannot exceed daily_budget_usd"
            )
        return self


def load_config() -> ProductionConfig:
    """Load and validate config. Call once at process startup.
    Raises pydantic.ValidationError with clear messages if anything is wrong.
    """
    cfg = ProductionConfig()
    return cfg
"""

(WORKSPACE / "auto_researcher" / "config_prod.py").parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "auto_researcher" / "config_prod.py").write_text(CONFIG_PY)

# ── Demo: show what happens with bad/missing config ──────────────────────────
from pydantic import ValidationError
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import field_validator, model_validator
from typing import Literal

class DemoConfig(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="DEMO_")
    anthropic_api_key: str
    api_key: str
    daily_budget_usd: float = 10.0

    @field_validator("anthropic_api_key")
    @classmethod
    def key_must_look_like_anthropic(cls, v: str) -> str:
        if not v.startswith("sk-ant-"):
            raise ValueError("Must start with 'sk-ant-'")
        return v

# Test 1: missing required key
import os
for k in ["DEMO_ANTHROPIC_API_KEY", "DEMO_API_KEY"]:
    os.environ.pop(k, None)

try:
    DemoConfig()
except ValidationError as e:
    print("❌ Missing required keys (expected):")
    for err in e.errors():
        print(f"   [{err['loc'][0]}] {err['msg']}")

# Test 2: wrong key format
os.environ["DEMO_ANTHROPIC_API_KEY"] = "wrong-format-key"
os.environ["DEMO_API_KEY"] = "a" * 20  # valid length

try:
    DemoConfig()
except ValidationError as e:
    print("\n❌ Bad key format (expected):")
    for err in e.errors():
        print(f"   [{err['loc'][0]}] {err['msg']}")

# Test 3: valid config
os.environ["DEMO_ANTHROPIC_API_KEY"] = "sk-ant-" + "x" * 40
cfg = DemoConfig()
print(f"\n✅ Valid config loaded: budget=${cfg.daily_budget_usd}/day")
print("   → If any required var is missing, the process CRASHES at startup, not mid-request.")

## 4 — Structured JSON logging + request IDs

### Why not `print()` or `logging.info()`?

```
# What print() produces in production logs:
2026-06-24 10:01:22 research request received
2026-06-24 10:01:23 calling LLM
2026-06-24 10:01:24 done

# Problems: Which request? Which user? How long? Which model? Error or success?
# You can't filter, alert, or graph this.

# What structured JSON logging produces:
{"ts":"2026-06-24T10:01:22Z","level":"INFO","event":"request_start",
 "request_id":"req-abc123","path":"/research","client_ip":"1.2.3.4"}
{"ts":"2026-06-24T10:01:23Z","level":"INFO","event":"llm_call",
 "request_id":"req-abc123","model":"claude-haiku-4-5","tokens":420}
{"ts":"2026-06-24T10:01:24Z","level":"INFO","event":"request_end",
 "request_id":"req-abc123","status":200,"latency_ms":1823,"cost_usd":0.0031}

# Now you can: grep by request_id, sum cost_usd, P95 latency_ms, alert on errors.
```

### Request ID threading

A `request_id` is generated per HTTP request and must flow through every log line,
every LLM call, and every downstream service call. The mechanism is Python's
`contextvars.ContextVar` — it's thread-safe AND async-safe.

In [ ]:
# ── Structured logging system ─────────────────────────────────────────────────
import contextvars
import sys
from typing import Optional

# ContextVar carries request_id through async call chains without passing it
# as a parameter to every function.
_request_id_var: contextvars.ContextVar[str] = contextvars.ContextVar(
    "request_id", default="-"
)


class JSONLogger:
    """Structured JSON logger. One instance per module, like standard logging."""

    def __init__(self, name: str):
        self.name = name

    def _emit(self, level: str, event: str, **kwargs) -> None:
        record = {
            "ts": datetime.utcnow().isoformat() + "Z",
            "level": level,
            "logger": self.name,
            "request_id": _request_id_var.get(),
            "event": event,
            **kwargs,
        }
        # Print to stdout — container orchestrators capture stdout as logs
        print(json.dumps(record), flush=True)

    def info(self, event: str, **kwargs):    self._emit("INFO",    event, **kwargs)
    def warning(self, event: str, **kwargs): self._emit("WARNING", event, **kwargs)
    def error(self, event: str, **kwargs):   self._emit("ERROR",   event, **kwargs)
    def debug(self, event: str, **kwargs):   self._emit("DEBUG",   event, **kwargs)


def get_logger(name: str) -> JSONLogger:
    return JSONLogger(name)


# ── Demo: what logs look like ─────────────────────────────────────────────────
logger = get_logger("demo")

# Simulate a request coming in
token = _request_id_var.set(f"req-{uuid.uuid4().hex[:8]}")

logger.info("request_start", path="/research", client_ip="1.2.3.4", method="POST")
time.sleep(0.01)
logger.info("llm_call", model="claude-haiku-4-5", input_tokens=420, output_tokens=150)
time.sleep(0.01)
logger.info("request_end", status=200, latency_ms=823, cost_usd=0.0031)

# Reset (in FastAPI middleware, this happens automatically per-request)
_request_id_var.reset(token)

# Simulate an error request with a different request_id
token2 = _request_id_var.set(f"req-{uuid.uuid4().hex[:8]}")
logger.info("request_start", path="/research", client_ip="5.6.7.8", method="POST")
logger.error("llm_call_failed", model="claude-sonnet-4-5", error="RateLimitError", retry=1)
logger.info("request_end", status=429, latency_ms=1200, cost_usd=0.0)
_request_id_var.reset(token2)

print()
print("↑ Every log line has: ts, level, logger, request_id, event + custom fields")
print("  Cloud Log parsers (Datadog, GCP, CloudWatch) can now:")
print("  - GROUP all logs for a single request_id")
print("  - ALERT on level=ERROR")
print("  - GRAPH P95 of latency_ms")
print("  - SUM cost_usd per day")

## 5 — FastAPI production middleware stack

A production FastAPI app has layers of middleware. Each request passes through
ALL of them (outermost first for requests, innermost first for responses):

```
                    ┌─────────────────────────────────┐
  HTTP Request ───► │  1. RequestID   (add X-Request-ID)│
                    │  2. Auth        (check API key)   │
                    │  3. RateLimiter (token bucket)    │
                    │  4. BudgetGuard (daily cap)       │
                    │  5. Route handler                 │
                    └─────────────────────────────────┘
```

### Token bucket rate limiter

```
Each IP gets a "bucket" that starts full (capacity = rate_limit_rpm).
Each request consumes 1 token.
Tokens refill at rate_limit_rpm / 60 per second.
If bucket is empty → 429 Too Many Requests.

Why token bucket vs "count per minute"?
  - Count per minute: client can send 60 req in first second → spike
  - Token bucket:     client can burst up to capacity, then rate-limited
  - More fair and protects against thundering herds
```

In [ ]:
# ── Production FastAPI middleware stack ───────────────────────────────────────
MIDDLEWARE_PY = '''
import time, uuid, json
from datetime import datetime
from typing import Dict, Optional
from contextvars import ContextVar
from dataclasses import dataclass, field
from fastapi import Request, Response
from fastapi.responses import JSONResponse
from starlette.middleware.base import BaseHTTPMiddleware

# Shared ContextVar for request_id (defined in logging module in real project)
_request_id_var: ContextVar[str] = ContextVar("request_id", default="-")


# ── Middleware 1: Request ID ──────────────────────────────────────────────────
class RequestIDMiddleware(BaseHTTPMiddleware):
    """Generate or propagate a request ID. Attach to context + response header."""

    async def dispatch(self, request: Request, call_next):
        # Honour an upstream-supplied ID (e.g. from Nginx or a gateway)
        rid = request.headers.get("X-Request-ID") or f"req-{uuid.uuid4().hex[:12]}"
        token = _request_id_var.set(rid)
        try:
            response = await call_next(request)
            response.headers["X-Request-ID"] = rid
            return response
        finally:
            _request_id_var.reset(token)


# ── Middleware 2: API Key Auth ────────────────────────────────────────────────
class APIKeyAuthMiddleware(BaseHTTPMiddleware):
    """Simple API key check. Skip for /health, /ready, /live."""

    SKIP_PATHS = {"/health", "/ready", "/live", "/"}

    def __init__(self, app, api_key: str):
        super().__init__(app)
        self._api_key = api_key

    async def dispatch(self, request: Request, call_next):
        if request.url.path in self.SKIP_PATHS:
            return await call_next(request)

        provided = request.headers.get("X-API-Key", "")
        if not provided or provided != self._api_key:
            return JSONResponse(
                status_code=401,
                content={"error": "invalid_api_key",
                         "request_id": _request_id_var.get()},
            )
        return await call_next(request)


# ── Middleware 3: Token-bucket rate limiter ───────────────────────────────────
@dataclass
class _Bucket:
    tokens: float
    last_refill: float = field(default_factory=time.monotonic)


class RateLimitMiddleware(BaseHTTPMiddleware):
    """Per-IP token bucket. capacity=rpm, refill_rate=rpm/60 tokens/sec."""

    SKIP_PATHS = {"/health", "/ready", "/live", "/"}

    def __init__(self, app, rpm: int = 60):
        super().__init__(app)
        self._capacity = float(rpm)
        self._refill_rate = rpm / 60.0   # tokens per second
        self._buckets: Dict[str, _Bucket] = {}

    def _get_bucket(self, ip: str) -> _Bucket:
        if ip not in self._buckets:
            self._buckets[ip] = _Bucket(tokens=self._capacity)
        return self._buckets[ip]

    def _consume(self, ip: str) -> bool:
        """Return True if a token was consumed, False if rate limit exceeded."""
        bucket = self._get_bucket(ip)
        now = time.monotonic()
        elapsed = now - bucket.last_refill
        bucket.tokens = min(self._capacity, bucket.tokens + elapsed * self._refill_rate)
        bucket.last_refill = now
        if bucket.tokens >= 1.0:
            bucket.tokens -= 1.0
            return True
        return False

    async def dispatch(self, request: Request, call_next):
        if request.url.path in self.SKIP_PATHS:
            return await call_next(request)

        ip = request.client.host if request.client else "unknown"
        if not self._consume(ip):
            return JSONResponse(
                status_code=429,
                content={"error": "rate_limit_exceeded",
                         "request_id": _request_id_var.get()},
                headers={"Retry-After": "5"},
            )
        return await call_next(request)
'''

(WORKSPACE / "auto_researcher" / "middleware.py").write_text(MIDDLEWARE_PY)
print("✅ middleware.py written")

# ── Demo: rate limiter in isolation ──────────────────────────────────────────
import time
from dataclasses import dataclass, field as dc_field

@dataclass
class Bucket:
    tokens: float
    last_refill: float = dc_field(default_factory=time.monotonic)

def make_limiter(rpm: int = 10):
    capacity = float(rpm)
    refill_rate = rpm / 60.0
    buckets: dict[str, Bucket] = {}

    def consume(ip: str) -> tuple[bool, float]:
        if ip not in buckets:
            buckets[ip] = Bucket(tokens=capacity)
        b = buckets[ip]
        now = time.monotonic()
        b.tokens = min(capacity, b.tokens + (now - b.last_refill) * refill_rate)
        b.last_refill = now
        if b.tokens >= 1.0:
            b.tokens -= 1.0
            return True, b.tokens
        return False, b.tokens

    return consume

consume = make_limiter(rpm=5)  # 5 requests per minute for demo

print("Token bucket demo (capacity=5, rapid-fire 8 requests):")
for i in range(8):
    ok, remaining = consume("1.2.3.4")
    status = "✅ allowed" if ok else "❌ rate limited"
    print(f"  Request {i+1}: {status}  (tokens remaining: {remaining:.1f})")

print()
print("After 3-second wait (refill ~0.25 tokens/sec × 3 = 0.75 tokens):")
time.sleep(3)
ok, remaining = consume("1.2.3.4")
print(f"  Request 9: {'✅ allowed' if ok else '❌ rate limited'}  (tokens remaining: {remaining:.1f})")

## 6 — Health checks: /health vs /ready vs /live

Kubernetes (and Fly.io, Railway, ECS) support three probe types. They have
**different consequences** when they fail:

| Endpoint | Probe | Fails → | Checks |
|----------|-------|---------|--------|
| `/live`  | Liveness | Pod **restarted** | Is the process alive? (no deadlock) |
| `/ready` | Readiness | Pod **removed from load balancer** | Is the app ready to serve traffic? |
| `/health`| Manual / monitoring | Alert fired | Full diagnostic snapshot |

### Design rules

- `/live` must be **extremely cheap** — just return 200. It runs every 10 seconds.
  If it does anything expensive and times out, Kubernetes will restart your pod forever.
- `/ready` can check that the retrieval index is loaded, DB connection is alive, etc.
  Failing `/ready` is safe — traffic just goes to other pods.
- `/health` is for monitoring dashboards. Can return detailed diagnostics.

**Common mistake:** putting a database query in `/live`. The pod gets restarted because
the DB is slow during a backup window — unrelated to whether the pod is alive.

In [ ]:
# ── Health check system ───────────────────────────────────────────────────────
HEALTH_PY = '''
import time, os
from typing import Dict, Any, Optional
from dataclasses import dataclass
from fastapi import APIRouter, Response
from fastapi.responses import JSONResponse

router = APIRouter()

# Global startup time (set when app starts)
_start_time = time.monotonic()
_retrieval_store_ready = False   # set True after index is loaded
_daily_cost_usd = 0.0            # updated by BudgetGuard middleware


# ── /live: liveness probe ─────────────────────────────────────────────────────
@router.get("/live", include_in_schema=False)
async def liveness():
    """Kubernetes liveness probe. Returns 200 if process is not deadlocked.
    KEEP THIS TRIVIAL — any failure here restarts the pod.
    """
    return {"status": "alive"}


# ── /ready: readiness probe ───────────────────────────────────────────────────
@router.get("/ready", include_in_schema=False)
async def readiness(response: Response):
    """Kubernetes readiness probe. Returns 200 only when ready to serve traffic.
    Failure (503) removes this pod from the load balancer rotation.
    """
    checks: Dict[str, bool] = {
        "retrieval_store": _retrieval_store_ready,
        "anthropic_key_set": bool(os.getenv("ANTHROPIC_API_KEY")
                                   or os.getenv("AR_ANTHROPIC_API_KEY")),
    }
    all_ok = all(checks.values())
    response.status_code = 200 if all_ok else 503
    return {"status": "ready" if all_ok else "not_ready", "checks": checks}


# ── /health: full diagnostic snapshot ────────────────────────────────────────
@router.get("/health")
async def health():
    """Full health report for monitoring dashboards (Grafana, Datadog).
    Returns 200 even when degraded — callers should inspect 'status' field.
    """
    uptime_s = time.monotonic() - _start_time
    return {
        "status": "healthy",
        "version": os.getenv("APP_VERSION", "dev"),
        "environment": os.getenv("AR_ENVIRONMENT", "unknown"),
        "uptime_seconds": round(uptime_s, 1),
        "components": {
            "retrieval_store": {
                "status": "ok" if _retrieval_store_ready else "initializing"
            },
            "budget": {
                "daily_spent_usd": round(_daily_cost_usd, 4),
                "daily_limit_usd": float(os.getenv("AR_DAILY_BUDGET_USD", "10.0")),
                "status": "ok" if _daily_cost_usd < float(
                    os.getenv("AR_DAILY_BUDGET_USD", "10.0")
                ) else "exhausted",
            },
        },
    }
'''

(WORKSPACE / "auto_researcher" / "health.py").write_text(HEALTH_PY)
print("✅ health.py written")

# ── Simulate what probe responses look like ───────────────────────────────────
import random

def simulate_probes(retrieval_ready: bool, budget_ok: bool):
    liveness = {"status": "alive"}  # always 200

    readiness_checks = {
        "retrieval_store": retrieval_ready,
        "anthropic_key_set": True,
    }
    all_ready = all(readiness_checks.values())
    readiness = {
        "http_status": 200 if all_ready else 503,
        "body": {"status": "ready" if all_ready else "not_ready", "checks": readiness_checks}
    }

    health = {
        "status": "healthy" if (retrieval_ready and budget_ok) else "degraded",
        "uptime_seconds": 42.3,
        "components": {
            "retrieval_store": {"status": "ok" if retrieval_ready else "initializing"},
            "budget": {"daily_spent_usd": 0.0 if budget_ok else 11.5,
                       "daily_limit_usd": 10.0,
                       "status": "ok" if budget_ok else "exhausted"},
        },
    }
    return liveness, readiness, health

print("Scenario 1: app just started (retrieval store still initializing)")
live, ready, health = simulate_probes(retrieval_ready=False, budget_ok=True)
print(f"  /live   → HTTP 200: {live}")
print(f"  /ready  → HTTP {ready['http_status']}: {ready['body']['status']} (pod NOT in LB rotation)")
print(f"  /health → {health['status']}")

print()
print("Scenario 2: fully operational")
live, ready, health = simulate_probes(retrieval_ready=True, budget_ok=True)
print(f"  /live   → HTTP 200: {live}")
print(f"  /ready  → HTTP {ready['http_status']}: {ready['body']['status']} (pod IN LB rotation)")
print(f"  /health → {health['status']}")

print()
print("Scenario 3: daily budget exhausted")
live, ready, health = simulate_probes(retrieval_ready=True, budget_ok=False)
print(f"  /live   → HTTP 200: {live}  ← process is alive, DON'T restart")
print(f"  /ready  → HTTP {ready['http_status']}: {ready['body']['status']}  ← still serves (returns 402 from BudgetGuard)")
print(f"  /health → {health['status']}  ← monitoring alert fires")

## 7 — Graceful shutdown: drain before dying

When Kubernetes rolls out a new version, it sends `SIGTERM` to the old pod.  
What happens next depends on whether you handle it:

```
WITHOUT graceful shutdown         WITH graceful shutdown
──────────────────────────        ──────────────────────────────────────
SIGTERM received                  SIGTERM received
  → process exits immediately     → /ready returns 503 (stops new traffic)
  → all in-flight requests        → wait up to 30s for in-flight to finish
     get 502 Bad Gateway          → in-flight requests complete normally
                                  → process exits cleanly

User sees: random 502 errors      User sees: zero errors during rollout
```

### FastAPI lifespan

FastAPI's `lifespan` context manager handles startup AND shutdown in one place:

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    # ── STARTUP: runs before first request ──
    await load_retrieval_index()
    logger.info("startup_complete")

    yield  # app serves requests here

    # ── SHUTDOWN: runs after last request drains ──
    await cleanup_connections()
    logger.info("shutdown_complete")
```

The `--timeout-graceful-shutdown 30` flag we set in CMD tells uvicorn to
wait up to 30 seconds for the `yield` body to finish before force-killing.

In [ ]:
# ── Full production app wiring ─────────────────────────────────────────────────
APP_PY = '''
import asyncio, logging, time, uuid
from contextlib import asynccontextmanager
from typing import Optional
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

# These would import from your project modules:
# from .config_prod import load_config
# from .middleware import RequestIDMiddleware, APIKeyAuthMiddleware, RateLimitMiddleware
# from .health import router as health_router
# from .budget import BudgetGuardMiddleware

# ── Startup / shutdown ────────────────────────────────────────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    # ─── STARTUP ─────────────────────────────────────────────────────────────
    logger = logging.getLogger("auto_researcher")

    # 1. Load and validate config (fails fast if secrets missing)
    # config = load_config()  # raises ValidationError → process exits before serving
    logger.info("config_loaded")

    # 2. Load retrieval index (warm it up before traffic hits)
    # await app.state.retrieval.warm_up()
    logger.info("retrieval_index_warm")

    # 3. Signal readiness (mark /ready as OK)
    # from . import health; health._retrieval_store_ready = True
    logger.info("startup_complete",
                extra={"event": "startup_complete", "ts": time.time()})

    yield  # ─── APP SERVES REQUESTS HERE ─────────────────────────────────────

    # ─── SHUTDOWN ────────────────────────────────────────────────────────────
    logger.info("shutdown_initiated")

    # Give in-flight requests up to 30s (uvicorn --timeout-graceful-shutdown)
    # This code runs AFTER uvicorn stops accepting new connections.

    # Close any open DB connections, flush metric buffers, etc.
    # await app.state.retrieval.close()
    # await app.state.eval_pipeline.drain(limit=10)  # flush pending evals

    logger.info("shutdown_complete")


# ── App factory ───────────────────────────────────────────────────────────────
def create_app() -> FastAPI:
    app = FastAPI(
        title="auto-researcher",
        version="2.0.0",
        description="AI-powered research agent API",
        lifespan=lifespan,
        # Disable docs in production
        docs_url=None if False else "/docs",  # set to None when env=production
        redoc_url=None,
    )

    # ── Middleware (order matters: applied bottom-up, executed top-down) ──────
    # app.add_middleware(BudgetGuardMiddleware, daily_budget_usd=config.daily_budget_usd)
    # app.add_middleware(RateLimitMiddleware, rpm=config.rate_limit_rpm)
    # app.add_middleware(APIKeyAuthMiddleware, api_key=config.api_key)
    # app.add_middleware(RequestIDMiddleware)

    # ── Routes ───────────────────────────────────────────────────────────────
    # app.include_router(health_router)
    # app.include_router(research_router, prefix="/api/v1")

    return app


app = create_app()


# ── Request / response models ─────────────────────────────────────────────────
class ResearchRequest(BaseModel):
    query: str
    max_cost_usd: Optional[float] = None  # override per-request budget


class ResearchResponse(BaseModel):
    answer: str
    cost_usd: float
    latency_ms: int
    request_id: str
    model: str


@app.post("/research", response_model=ResearchResponse)
async def research(req: ResearchRequest, request: Request):
    start = time.monotonic()
    # result = await app.state.pipeline.research(req.query)
    # Placeholder:
    result = {"answer": "Demo answer", "cost_usd": 0.003, "model": "claude-haiku-4-5"}
    return ResearchResponse(
        answer=result["answer"],
        cost_usd=result["cost_usd"],
        latency_ms=int((time.monotonic() - start) * 1000),
        request_id=request.headers.get("X-Request-ID", "-"),
        model=result["model"],
    )
'''

(WORKSPACE / "auto_researcher" / "api.py").write_text(APP_PY)
print("✅ api.py written")

# ── Show graceful shutdown simulation ─────────────────────────────────────────
print()
print("Graceful shutdown sequence simulation:")
print()

async def simulate_shutdown():
    # Track in-flight requests
    in_flight: list[asyncio.Task] = []

    async def fake_request(n: int, duration: float):
        print(f"  → Request {n} started (will take {duration:.1f}s)")
        await asyncio.sleep(duration)
        print(f"  ← Request {n} completed ✅")

    # Start 3 requests with varying durations
    for i, dur in enumerate([0.5, 1.2, 0.3], 1):
        task = asyncio.create_task(fake_request(i, dur))
        in_flight.append(task)

    await asyncio.sleep(0.1)
    print()
    print("  [SIGTERM received]")
    print("  uvicorn stops accepting NEW connections")
    print("  waiting for in-flight requests to drain (timeout=30s)...")
    print()

    # Wait for all in-flight to complete
    await asyncio.gather(*in_flight)

    print()
    print("  lifespan cleanup running (flush DB, close connections)...")
    await asyncio.sleep(0.1)
    print("  [process exits cleanly] ✅")

asyncio.run(simulate_shutdown())

## 8 — BudgetGuard middleware: enforce a daily spend cap

Your `auto_researcher` calls Anthropic. Each call costs money.  
A bug (infinite loop, misconfigured client, DDoS) can generate a huge bill.

**BudgetGuard is your last line of defense:**

```
Every request:
  1. Check today's cumulative spend in SQLite
  2. If spend >= daily_budget → return 402 Payment Required immediately
     (don't even call the LLM)
  3. After request completes, add cost_usd to today's spend

At midnight (UTC): spend resets to 0
```

**Why SQLite and not in-memory?** — A pod restart (deploy, crash, OOM kill)
would reset an in-memory counter, letting a bug run up a bill across restarts.
SQLite persists to the filesystem (or a mounted volume in Kubernetes).

In [ ]:
# ── BudgetGuard ───────────────────────────────────────────────────────────────
import sqlite3
from datetime import date, datetime
from threading import Lock


class BudgetGuard:
    """Thread-safe daily budget enforcer backed by SQLite."""

    def __init__(self, db_path: str = ":memory:", daily_limit_usd: float = 10.0):
        self._limit = daily_limit_usd
        self._lock = Lock()
        self._conn = sqlite3.connect(db_path, check_same_thread=False)
        self._conn.execute("""
            CREATE TABLE IF NOT EXISTS daily_spend (
                day TEXT PRIMARY KEY,
                cost_usd REAL NOT NULL DEFAULT 0.0
            )
        """)
        self._conn.commit()

    def _today(self) -> str:
        return date.today().isoformat()   # "2026-06-24"

    def get_today_spend(self) -> float:
        row = self._conn.execute(
            "SELECT cost_usd FROM daily_spend WHERE day = ?", (self._today(),)
        ).fetchone()
        return row[0] if row else 0.0

    def check(self) -> tuple[bool, float]:
        """Return (allowed, current_spend). Thread-safe."""
        with self._lock:
            spend = self.get_today_spend()
            return spend < self._limit, spend

    def record(self, cost_usd: float) -> float:
        """Add cost_usd to today's spend. Returns new total."""
        with self._lock:
            today = self._today()
            self._conn.execute("""
                INSERT INTO daily_spend (day, cost_usd) VALUES (?, ?)
                ON CONFLICT(day) DO UPDATE SET cost_usd = cost_usd + excluded.cost_usd
            """, (today, cost_usd))
            self._conn.commit()
            return self.get_today_spend()

    def status(self) -> dict:
        spend = self.get_today_spend()
        return {
            "day": self._today(),
            "spent_usd": round(spend, 4),
            "limit_usd": self._limit,
            "remaining_usd": round(max(0.0, self._limit - spend), 4),
            "utilization_pct": round(spend / self._limit * 100, 1),
        }


# ── Demo ──────────────────────────────────────────────────────────────────────
guard = BudgetGuard(daily_limit_usd=1.00)   # $1/day limit for demo

print("BudgetGuard demo (limit = $1.00/day)")
print()

# Simulate 12 research requests at $0.10 each
for i in range(12):
    allowed, spend = guard.check()
    if not allowed:
        print(f"  Request {i+1:2d}: ❌ 402 Budget exhausted (${spend:.2f} spent, limit $1.00)")
        continue

    # Simulate the request succeeding and costing $0.10
    new_total = guard.record(0.10)
    print(f"  Request {i+1:2d}: ✅ served  (today's total: ${new_total:.2f})")

print()
print("Final status:")
import json as _json
print(_json.dumps(guard.status(), indent=2))
print()
print("💡 EXPERIMENT: Change daily_limit_usd=0.30 and see which request first gets rejected.")
print("   In the real app, BudgetGuard.record() is called by the pipeline AFTER each LLM call.")

## 9 — Deployment: Fly.io and Railway

Two platforms that make deploying a Docker container fast and cheap:

| | Fly.io | Railway |
|--|--------|----------|
| Free tier | Shared-CPU-1x, 256MB RAM | $5 credit/mo |
| Deploy command | `fly deploy` | `railway up` |
| Secrets | `fly secrets set KEY=val` | `railway variables set KEY=val` |
| Auto-HTTPS | ✅ | ✅ |
| Regions | 35+ | 3 |
| Health probes | Reads `/health` | Reads `/health` |
| Scale to zero | ✅ (Machines) | ✅ |
| Best for | Global edge | Quick prototypes |

### Deploy flow

```bash
# Fly.io
fly auth login
fly launch --no-deploy         # generates fly.toml
fly secrets set AR_ANTHROPIC_API_KEY=sk-ant-... AR_API_KEY=my-secret-key
fly deploy                     # builds, pushes, deploys
fly logs                       # tail logs

# Railway
npm install -g @railway/cli
railway login
railway init
railway variables set AR_ANTHROPIC_API_KEY=sk-ant-... AR_API_KEY=my-secret-key
railway up                     # deploys from current directory
```

In [ ]:
# ── Generate deployment configs ────────────────────────────────────────────────

FLY_TOML = """\
# fly.toml — auto-researcher v2 deployment config
# Deploy with: fly deploy

app = "auto-researcher-v2"      # Change to your app name
primary_region = "iad"           # us-east-1 (closest to most users)

[build]
  # Fly builds from the Dockerfile in the project root
  dockerfile = "Dockerfile"

[env]
  AR_ENVIRONMENT = "production"
  AR_LOG_LEVEL   = "INFO"
  AR_PORT        = "8000"
  AR_DAILY_BUDGET_USD     = "10.0"
  AR_PER_REQUEST_BUDGET_USD = "0.10"
  AR_RATE_LIMIT_RPM       = "60"
  APP_VERSION             = "2.0.0"
  # Secrets (NOT in this file) — set with: fly secrets set AR_ANTHROPIC_API_KEY=...
  # AR_ANTHROPIC_API_KEY  <- fly secrets
  # AR_API_KEY            <- fly secrets

[http_service]
  internal_port = 8000
  force_https   = true
  auto_stop_machines  = true    # scale-to-zero when idle
  auto_start_machines = true    # wake on traffic
  min_machines_running = 0

  [http_service.concurrency]
    type       = "requests"
    soft_limit = 25
    hard_limit = 50

[[vm]]
  cpu_kind = "shared"
  cpus     = 1
  memory   = "512mb"   # bump to 1gb if you use a local embedding model

[checks]
  [checks.health]
    grace_period = "15s"   # wait before first check (let lifespan startup finish)
    interval     = "30s"
    method       = "GET"
    path         = "/ready"   # /ready not /health — we want LB probe semantics
    timeout      = "5s"
"""

RAILWAY_JSON = """\
{
  "$schema": "https://railway.app/railway.schema.json",
  "build": {
    "builder": "DOCKERFILE",
    "dockerfilePath": "./Dockerfile"
  },
  "deploy": {
    "startCommand": "uvicorn auto_researcher.api:app --host 0.0.0.0 --port $PORT --timeout-graceful-shutdown 30",
    "healthcheckPath": "/ready",
    "healthcheckTimeout": 300,
    "restartPolicyType": "ON_FAILURE",
    "restartPolicyMaxRetries": 3
  }
}
"""

DOTENV_EXAMPLE = """\
# .env.example — copy to .env and fill in values
# NEVER commit .env to git (it's in .gitignore)

# Required secrets
AR_ANTHROPIC_API_KEY=sk-ant-YOUR_KEY_HERE
AR_API_KEY=generate-with-openssl-rand-hex-32

# Optional overrides
AR_ENVIRONMENT=development
AR_LOG_LEVEL=DEBUG
AR_DAILY_BUDGET_USD=2.0
AR_PER_REQUEST_BUDGET_USD=0.10
AR_RATE_LIMIT_RPM=60
"""

(WORKSPACE / "fly.toml").write_text(FLY_TOML)
(WORKSPACE / "railway.json").write_text(RAILWAY_JSON)
(WORKSPACE / ".env.example").write_text(DOTENV_EXAMPLE)

print("✅ Deployment files written:")
print("   fly.toml       — Fly.io config")
print("   railway.json   — Railway config")
print("   .env.example   — environment variable template")
print()
print("To deploy to Fly.io:")
print("  1. pip install flyctl   OR   brew install flyctl")
print("  2. fly auth login")
print("  3. fly launch --no-deploy   # creates app, skip first deploy")
print("  4. fly secrets set AR_ANTHROPIC_API_KEY=sk-ant-... AR_API_KEY=$(openssl rand -hex 16)")
print("  5. fly deploy")
print("  6. fly open")
print()
print("To deploy to Railway:")
print("  1. npm install -g @railway/cli")
print("  2. railway login")
print("  3. railway init")
print("  4. railway variables set AR_ANTHROPIC_API_KEY=sk-ant-... AR_API_KEY=$(openssl rand -hex 16)")
print("  5. railway up")

## 10 — Load testing: know your limits before traffic does

A load test answers two questions:
1. **What's my throughput ceiling?** (requests/sec at acceptable P95 latency)
2. **Does my system fail gracefully?** (rate limiter fires, 429s returned, no crashes)

### Load test plan for auto_researcher_v2

```
Phase 1 — baseline (5 req/sec for 30s):
  → Expect P50 ~1.5s, P95 ~3s, 0% errors

Phase 2 — ramp (5 → 60 req/sec over 60s):
  → Watch for P95 to climb, rate limiter to fire

Phase 3 — sustained peak (60 req/sec for 30s):
  → Expect ~rate_limit_rpm% of requests to get 429
  → No crashes, no 500s

Phase 4 — recovery (back to 5 req/sec):
  → P95 should return to baseline
  → Token buckets should be refilled
```

The load tester below hits a **mock server** (no real LLM calls, no $ spent)  
so you can validate the middleware stack without cost.

In [ ]:
# ── Async load tester ─────────────────────────────────────────────────────────
import asyncio, time, uuid, statistics, json
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class RequestResult:
    status: int
    latency_ms: float
    request_id: str = ""
    error: Optional[str] = None


@dataclass
class LoadTestReport:
    total: int
    results: List[RequestResult]
    duration_s: float

    @property
    def throughput(self) -> float:
        return self.total / self.duration_s

    @property
    def status_counts(self) -> dict:
        counts: dict[int, int] = {}
        for r in self.results:
            counts[r.status] = counts.get(r.status, 0) + 1
        return dict(sorted(counts.items()))

    def percentile(self, p: float) -> float:
        latencies = sorted(r.latency_ms for r in self.results if r.status == 200)
        if not latencies:
            return 0.0
        idx = int(len(latencies) * p / 100)
        return latencies[min(idx, len(latencies) - 1)]

    def print_summary(self, label: str = ""):
        print(f"\n{'─'*50}")
        if label:
            print(f"  {label}")
        print(f"  Total requests : {self.total}")
        print(f"  Duration       : {self.duration_s:.1f}s")
        print(f"  Throughput     : {self.throughput:.1f} req/s")
        print(f"  Status codes   : {self.status_counts}")
        latencies_200 = [r.latency_ms for r in self.results if r.status == 200]
        if latencies_200:
            print(f"  P50 latency    : {self.percentile(50):.0f} ms")
            print(f"  P95 latency    : {self.percentile(95):.0f} ms")
            print(f"  P99 latency    : {self.percentile(99):.0f} ms")
        error_rate = sum(1 for r in self.results if r.status >= 400) / self.total * 100
        print(f"  Error rate     : {error_rate:.1f}% (≥400 status)")
        rate_limited = sum(1 for r in self.results if r.status == 429)
        print(f"  Rate limited   : {rate_limited} ({rate_limited/self.total*100:.0f}%)")
        print(f"{'─'*50}")


# ── Mock server (replaces the real API for load testing) ──────────────────────
# This simulates the middleware stack without making LLM calls.
# In real load testing, you'd point this at a staging deployment.

class MockServer:
    """Simulates our production middleware stack."""

    def __init__(self, rpm: int = 60, base_latency_ms: float = 50.0):
        self._rpm = rpm
        self._base_latency_ms = base_latency_ms
        # Per-IP token buckets
        self._buckets: dict[str, float] = {}
        self._last_refill: dict[str, float] = {}

    def _rate_check(self, ip: str) -> bool:
        now = time.monotonic()
        capacity = float(self._rpm)
        rate = self._rpm / 60.0
        if ip not in self._buckets:
            self._buckets[ip] = capacity
            self._last_refill[ip] = now
        elapsed = now - self._last_refill[ip]
        self._buckets[ip] = min(capacity, self._buckets[ip] + elapsed * rate)
        self._last_refill[ip] = now
        if self._buckets[ip] >= 1.0:
            self._buckets[ip] -= 1.0
            return True
        return False

    async def handle(self, ip: str = "1.2.3.4") -> RequestResult:
        import random
        t0 = time.monotonic()

        if not self._rate_check(ip):
            return RequestResult(status=429, latency_ms=(time.monotonic()-t0)*1000)

        # Simulate LLM latency (lognormal around base_latency)
        latency = random.lognormvariate(
            mu=0, sigma=0.4) * self._base_latency_ms / 1000.0
        await asyncio.sleep(latency)

        return RequestResult(
            status=200,
            latency_ms=(time.monotonic()-t0)*1000,
            request_id=f"req-{uuid.uuid4().hex[:8]}",
        )


async def run_load_test(
    server: MockServer,
    target_rps: float,
    duration_s: float,
    max_concurrent: int = 20,
) -> LoadTestReport:
    results: List[RequestResult] = []
    sem = asyncio.Semaphore(max_concurrent)
    interval = 1.0 / target_rps
    t_start = time.monotonic()

    async def one_request():
        async with sem:
            r = await server.handle()
            results.append(r)

    tasks = []
    while time.monotonic() - t_start < duration_s:
        tasks.append(asyncio.create_task(one_request()))
        await asyncio.sleep(interval)

    await asyncio.gather(*tasks, return_exceptions=True)
    return LoadTestReport(
        total=len(results),
        results=results,
        duration_s=time.monotonic() - t_start,
    )


# ── Run the load test ─────────────────────────────────────────────────────────
print("Running load test against mock server (rate limit = 60 rpm = 1 req/sec)")
print("(Using mock server: no real LLM calls, no API cost)")

server = MockServer(rpm=60, base_latency_ms=150)

async def run_all_phases():
    # Phase 1: below rate limit
    report1 = await run_load_test(server, target_rps=0.5, duration_s=6)
    report1.print_summary("Phase 1: 0.5 req/s (below rate limit 1 req/s)")

    # Phase 2: at rate limit
    report2 = await run_load_test(server, target_rps=1.0, duration_s=6)
    report2.print_summary("Phase 2: 1.0 req/s (at rate limit)")

    # Phase 3: above rate limit (expect 429s)
    report3 = await run_load_test(server, target_rps=3.0, duration_s=6)
    report3.print_summary("Phase 3: 3.0 req/s (3× over limit, expect 429s)")

asyncio.run(run_all_phases())

print()
print("Key observations:")
print("  Phase 1: All 200s — system well within capacity")
print("  Phase 2: All 200s — operating at limit but token bucket absorbs bursts")
print("  Phase 3: Mix of 200 and 429 — rate limiter protecting the LLM backend")
print()
print("💡 EXPERIMENT: Change rpm=10 in MockServer and rerun. At what Phase 3 rps")
print("   do you start seeing >50% 429s?")

## 11 — 10 Production Pitfalls

| # | Pitfall | What happens | Fix |
|---|---------|-------------|-----|
| 1 | **`CMD` uses shell form** | `CMD uvicorn ...` wraps in `/bin/sh -c`. SIGTERM goes to shell, not uvicorn. Requests get 502 on every deploy. | Use exec form: `CMD ["uvicorn", ...]` |
| 2 | **Liveness probe calls the DB** | DB slow during backup → probe times out → pod restarted → cascade | `/live` must return 200 instantly. DB check belongs in `/ready`. |
| 3 | **Config read inside request handler** | One typo in env var = every request fails with 500 | Call `load_config()` once at startup inside `lifespan()` |
| 4 | **`print()` for logging** | Logs unstructured, no timestamps, can't grep by request_id, can't alert on ERROR | Use JSONLogger with ContextVar-threaded request_id |
| 5 | **Single-stage Docker** | Image is 1.4 GB. Cold start takes 90s. Free tier OOMs during pull. | Multi-stage: runtime image ~120 MB |
| 6 | **Budget counter in-memory** | Pod restart (deploy/crash) resets counter. Bug can run up $ across restarts. | Store in SQLite on mounted volume |
| 7 | **Rate limiter per-instance** | Behind a load balancer with 3 replicas, each IP gets 3× the limit | Use Redis for shared rate limit state (or sticky sessions) |
| 8 | **No `--timeout-graceful-shutdown`** | `uvicorn` exits immediately on SIGTERM even though lifespan has cleanup code | Always set `--timeout-graceful-shutdown 30` |
| 9 | **Secrets in `fly.toml` / `railway.json`** | Config files get committed to git. Keys leaked. | Use `fly secrets set` / `railway variables set`. Never put secrets in config files. |
| 10 | **Load test against production** | Real LLM calls, real $, real latency to your users during the test | Always load test against a **staging** environment with a mock backend |

---

### The one rule for production AI APIs

> **Every dollar you spend is a dollar that must pass through your BudgetGuard.**  
> Never let a code path reach the LLM without first checking the budget, recording the cost after, and logging both with a request_id.

In [ ]:
# ── Verify all files were written ─────────────────────────────────────────────
import pathlib

WORKSPACE = pathlib.Path("/content/auto_researcher_v2")

expected = [
    "Dockerfile",
    "fly.toml",
    "railway.json",
    ".env.example",
    "auto_researcher/config_prod.py",
    "auto_researcher/middleware.py",
    "auto_researcher/health.py",
    "auto_researcher/api.py",
]

print("Phase 5 L53 file manifest:")
print()
all_ok = True
for f in expected:
    path = WORKSPACE / f
    ok = path.exists()
    all_ok = all_ok and ok
    size = path.stat().st_size if ok else 0
    print(f"  {'✅' if ok else '❌'} {f:<45} ({size:>5} bytes)")

print()
if all_ok:
    print("✅ All files present. auto_researcher_v2 is production-ready.")
else:
    print("❌ Some files missing — re-run the cells above.")

print()
print("Production readiness checklist:")
checklist = [
    ("Secrets never in source",            "✅  .env.example + fly secrets / railway variables"),
    ("Config fails fast at startup",        "✅  ProductionConfig with Pydantic validators"),
    ("Structured logs",                     "✅  JSONLogger with event+fields"),
    ("Request IDs",                         "✅  ContextVar-threaded across every log line"),
    ("Health endpoints",                    "✅  /live /ready /health with correct semantics"),
    ("Graceful shutdown",                   "✅  lifespan + --timeout-graceful-shutdown 30"),
    ("Rate limiting",                       "✅  Token bucket per IP in RateLimitMiddleware"),
    ("Auth",                                "✅  X-API-Key header in APIKeyAuthMiddleware"),
    ("Non-root Docker user",               "✅  USER appuser (UID 1000) in Dockerfile"),
    ("Resource limits",                     "✅  Fly.io VM config: 1 CPU, 512 MB RAM"),
    ("Budget cap",                          "✅  BudgetGuard with SQLite-backed daily limit"),
    ("Load-tested capacity",               "✅  Async load tester with phase ramp"),
]
for item, status in checklist:
    print(f"  {status} — {item}")

## 12 — Homework

1. **Deploy to Fly.io** — Follow the steps in §9 to deploy `auto_researcher_v2` to Fly.io's free tier.
   Hit your live `/health` and `/research` endpoints. Share the URL in your notes.

2. **Wire BudgetGuard into the pipeline** — In `auto_researcher/pipeline.py` (from L51),
   add a `BudgetGuard` instance. Call `guard.check()` before `_infer()` and `guard.record(cost_usd)`
   after. Write a test that sends requests until the guard fires a 402.

3. **Structured logging end-to-end** — Replace all `print()` calls in `auto_researcher/`
   with `get_logger(__name__).info(...)`. Verify that every log line contains `request_id`.

4. **Multi-IP load test** — Modify the load tester to rotate through 5 different client IPs
   (`1.2.3.1` through `1.2.3.5`). Does each IP get its own rate limit bucket?
   What's the total system throughput now vs single-IP?

5. **Kubernetes probe manifests** — Write a Kubernetes `Deployment` YAML for `auto_researcher_v2`
   with `livenessProbe` pointing at `/live` (10s interval) and `readinessProbe` pointing at `/ready`
   (15s delay, 30s interval). This is the same pattern used by every production Kubernetes deploy.

---

## Next lesson: L54 — Developer Experience

The code works. The server runs. Now we make it **a joy to use**:

- **CLI polish** — `auto-researcher research "query"` with rich progress bars, cost display,
  `--json` output mode, `--version`, shell completion
- **Python SDK** — `from auto_researcher import AutoResearcher; ar = AutoResearcher(); ar.research("...")`
  so other developers can embed the pipeline in their own code
- **Auto-generated API docs** — OpenAPI spec + Mintlify / Redoc static site
- **CONTRIBUTING.md + dev setup** — one-command dev environment so open-source contributors
  can submit PRs without fighting local setup

After L54, there's one lesson left: **L55 Capstone: Ship It** — PyPI publish, GitHub release,
changelog, and the open-source README that proves your work to the world.